# US Traffic Fatality Prediction — Data Profiling
**DATA 495: Data Science Capstone**  
**Carl Stolpe | UMGC | May 2026**

This notebook profiles the NHTSA FARS 2015–2016 dataset, documenting variable selection rationale, null rates, outliers, and data quality issues across the 20 key variables selected for modeling.

**Data source:** [US Traffic Fatality Records — Kaggle (NHTSA FARS)](https://www.kaggle.com/datasets/usdot/nhtsa-traffic-fatalities)

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded.')

## 2. Load Raw FARS Tables

The FARS dataset is organized into three relational tables joined on `ST_CASE` + `YEAR`:
- **accident** — crash-level attributes (52 columns)
- **vehicle** — vehicle-level attributes (65 columns)
- **person** — person-level attributes (67 columns)

In [ ]:
# Load accident and person tables for both years
acc_2015 = pd.read_csv('../data/raw/accident_2015.csv', low_memory=False)
per_2015 = pd.read_csv('../data/raw/person_2015.csv',   low_memory=False)
acc_2016 = pd.read_csv('../data/raw/accident_2016.csv', low_memory=False)
per_2016 = pd.read_csv('../data/raw/person_2016.csv',   low_memory=False)

print(f'Accident 2015: {acc_2015.shape}')
print(f'Person 2015:   {per_2015.shape}')
print(f'Accident 2016: {acc_2016.shape}')
print(f'Person 2016:   {per_2016.shape}')

## 3. Join Tables and Filter to Drivers

In [ ]:
def build_driver_dataset(acc_df, per_df, year):
    """Join accident and person tables, filter to drivers only."""
    acc_df = acc_df.copy()
    per_df = per_df.copy()
    acc_df['YEAR'] = year
    per_df['YEAR'] = year
    merged = per_df.merge(acc_df, on=['ST_CASE', 'YEAR'], how='inner')
    drivers = merged[merged['PER_TYP'] == 1].copy()
    return drivers

df_2015 = build_driver_dataset(acc_2015, per_2015, 2015)
df_2016 = build_driver_dataset(acc_2016, per_2016, 2016)
df_all  = pd.concat([df_2015, df_2016], ignore_index=True)

print(f'2015 driver records: {len(df_2015):,}')
print(f'2016 driver records: {len(df_2016):,}')
print(f'Combined:            {len(df_all):,}')

## 4. Variable Selection

20 variables were selected from the 184 raw fields based on domain relevance (per NHTSA FARS Analytical User's Manual), null rates, and variance.

In [ ]:
KEY_VARS = [
    'ST_CASE', 'STATE', 'MONTH', 'HOUR', 'LGT_COND', 'WEATHER',
    'FUNC_SYS', 'MAN_COLL', 'FATALS', 'DRUNK_DR',
    'BODY_TYP', 'MOD_YEAR', 'SPEEDREL',
    'AGE', 'SEX', 'PER_TYP', 'INJ_SEV', 'REST_USE', 'ALC_RES', 'DR_DRINK'
]

profile_df = df_all[[v for v in KEY_VARS if v in df_all.columns]].copy()
print(f'Profile dataset shape: {profile_df.shape}')

## 5. Null Rate Analysis

In [ ]:
null_rates = (profile_df.isnull().sum() / len(profile_df) * 100).round(2)
null_summary = pd.DataFrame({
    'Missing Count': profile_df.isnull().sum(),
    'Missing %': null_rates,
    'Dtype': profile_df.dtypes
}).sort_values('Missing %', ascending=False)

print('Null Rate Summary')
print('-' * 40)
print(null_summary.to_string())

## 6. Descriptive Statistics for Key Continuous Variables

In [ ]:
cont_vars = ['AGE', 'FATALS', 'HOUR', 'MONTH', 'DRUNK_DR']

# Remap sentinel codes before computing stats
stats_df = profile_df[cont_vars].copy()
stats_df['HOUR'] = stats_df['HOUR'].replace(99, np.nan)
stats_df['AGE']  = stats_df['AGE'].replace([998, 999], np.nan)

desc = stats_df.describe().T
desc.index = ['Driver Age (years)', 'Fatalities per Crash',
              'Hour of Crash (0-23)', 'Month of Crash (1-12)',
              'Drunk Drivers in Crash']
desc.index.name = 'Variable'

print('Table 1: Descriptive Statistics — Key Continuous Variables (2015-2016 FARS, N=101,562)')
print(desc[['count','mean','std','min','25%','50%','75%','max']].round(2).to_string())

## 7. Target Variable Distribution

In [ ]:
inj_counts = profile_df['INJ_SEV'].value_counts().sort_index()
inj_labels = {
    0: 'No Injury',
    1: 'Possible Injury',
    2: 'Suspected Minor',
    3: 'Suspected Serious',
    4: 'Fatal Injury',
    9: 'Unknown'
}

print('INJ_SEV Distribution (pre-binary recoding)')
print('-' * 40)
for code, count in inj_counts.items():
    label = inj_labels.get(code, f'Code {code}')
    pct = count / len(profile_df) * 100
    print(f'  {label:<25} {count:>7,}  ({pct:.1f}%)')

fatal_count    = (profile_df['INJ_SEV'] == 4).sum()
nonfatal_count = (profile_df['INJ_SEV'] != 4).sum()
print(f'\nBinary target — Fatal (1): {fatal_count:,} ({fatal_count/len(profile_df):.1%})')
print(f'Binary target — Non-Fatal (0): {nonfatal_count:,} ({nonfatal_count/len(profile_df):.1%})')

## 8. Sentinel Code Inventory

FARS uses numeric sentinel codes to represent unknown or inapplicable values. These must be remapped to NaN before modeling.

In [ ]:
sentinel_map = {
    'HOUR':    [99],
    'AGE':     [998, 999],
    'SEX':     [8, 9],
    'MOD_YEAR':[9999]
}

print('Sentinel Code Inventory')
print('-' * 50)
for col, codes in sentinel_map.items():
    if col in profile_df.columns:
        count = profile_df[col].isin(codes).sum()
        pct   = count / len(profile_df) * 100
        print(f'  {col:<12} codes {codes} → {count:,} records ({pct:.1f}%)')

## 9. ALC_RES Missing Rate — Feature Engineering Rationale

The alcohol test result field (`ALC_RES`) has approximately 60% missing values due to systematic variation in test administration practices across jurisdictions. Rather than imputing BAC values, two binary derived features were engineered: `ALC_TESTED` and `ALC_POSITIVE`.

In [ ]:
if 'ALC_RES' in profile_df.columns:
    alc_missing = profile_df['ALC_RES'].isnull().sum()
    alc_pct     = alc_missing / len(profile_df) * 100
    print(f'ALC_RES missing: {alc_missing:,} records ({alc_pct:.1f}%)')
    print('Decision: Engineer ALC_TESTED and ALC_POSITIVE binary features instead of imputing.')
    print(f'  ALC_TESTED  = 1 if test was administered (ALC_RES is not null)')
    print(f'  ALC_POSITIVE = 1 if BAC >= 0.08')

## 10. Data Profile Summary

| Variable | Table | Type | Null % | Key Issue |
|----------|-------|------|--------|----------|
| ST_CASE | Accident | ID | 0% | Combine with YEAR for uniqueness |
| STATE | Accident | Categorical | 0% | Requires FIPS lookup |
| MONTH | Accident | Integer | 0% | Cyclical encoding recommended |
| HOUR | Accident | Integer | ~2% | Code 99 = unknown → NaN |
| LGT_COND | Accident | Categorical | <1% | Grouped into daylight vs. dark |
| WEATHER | Accident | Categorical | <1% | Low-frequency categories consolidated |
| FUNC_SYS | Accident | Categorical | ~3% | Rural/urban distinction extracted |
| MAN_COLL | Accident | Categorical | ~2% | Code 0 = non-collision; valid category |
| FATALS | Accident | Count | 0% | Right-skewed; max = 13 |
| DRUNK_DR | Accident | Binary | 0% | Used as binary flag |
| AGE | Person | Integer | ~2% | Codes 998, 999 → NaN |
| SEX | Person | Categorical | <1% | Codes 8, 9 → NaN |
| INJ_SEV | Person | Ordinal | 0% | Binary recoded: fatal(4)=1, else=0 |
| REST_USE | Person | Categorical | ~5% | Missing not at random; domain imputation |
| ALC_RES | Person | Decimal | ~60% | ALC_TESTED and ALC_POSITIVE engineered |